Universidade do Vale do Itajaí<br>
PPGCA - Programa de Pós-Graduação em Computação Aplicada<br>
Aprendizado de Máquina<br>
Prof. Felipe Viel<br>
Conteúdo: Aprendizado Não Supervisionado<br>

# Aprendizado de Máquina Não Supervisionado com Python e Scikit-learn


Este notebook contém exemplos simples e com dados reais para:
K-Means, Clustering Hierárquico, DBSCAN, GMM, PCA, Spectral Clustering, ICA, t-SNE, UMAP,
Isolation Forest e Regras de Associação com Apriori.

### Bibliotecas

Esta célula importa as bibliotecas necessárias para a execução dos exemplos. Elas incluem ferramentas para manipulação de dados, visualização gráfica, implementação dos algoritmos e cálculo de métricas de avaliação.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.image import imread
from sklearn.datasets import make_blobs, make_moons, load_iris, load_wine, load_digits, make_classification
from sklearn.preprocessing import StandardScaler
import seaborn as sns



from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import urllib.request
from pathlib import Path
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.cluster import SpectralClustering, KMeans
from sklearn.manifold import SpectralEmbedding
from sklearn.ensemble import IsolationForest
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.neighbors import kneighbors_graph
from sklearn.datasets import load_sample_image
from skimage import img_as_float

from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score
)

from scipy.cluster.hierarchy import linkage, dendrogram

np.random.seed(42)
plt.rcParams["figure.figsize"] = (6, 5)


## Função para métricas de clustering

Função para aplicar todas as métricas de avaliação no modelo. Se atente aos parâmetros.

In [ ]:

from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score

def clustering_metrics(X, labels, y_true=None):
    if len(np.unique(labels)) < 2:
        return dict(Silhouette=np.nan, DaviesBouldin=np.nan,
                    CalinskiHarabasz=np.nan, ARI=np.nan)
    return dict(
        Silhouette=silhouette_score(X, labels),
        DaviesBouldin=davies_bouldin_score(X, labels),
        CalinskiHarabasz=calinski_harabasz_score(X, labels),
        ARI=adjusted_rand_score(y_true, labels) if y_true is not None else np.nan
    )


## K-Means

### Introdução

O algoritmo K-Means é uma das técnicas de agrupamento mais conhecidas do aprendizado não supervisionado. Seu objetivo é separar os dados em **K grupos**, de forma que os pontos dentro de um mesmo grupo sejam semelhantes entre si e diferentes dos pontos de outros grupos.

A ideia central é bastante intuitiva:

1. Escolher o número de clusters `K`.
2. Inicializar `K` centróides.
3. Associar cada ponto ao centróide mais próximo.
4. Recalcular a posição dos centróides.
5. Repetir até que os centróides parem de se mover.

Neste exemplo, você verá:
- a implementação passo a passo do algoritmo;
- a aplicação com a biblioteca scikit-learn;
- o uso em segmentação de imagens.

**Aplicações práticas:** segmentação de clientes, compressão de imagens, agrupamento de documentos e análise exploratória de dados.

#### Dados sintéticos

Gera um conjunto de dados sintético utilizando `make_blobs()`. Cada blob representa um grupo de pontos distribuídos ao redor de um centróide. Esse tipo de dataset é ideal para demonstrar algoritmos de clustering, pois a estrutura dos grupos é conhecida previamente.

In [ ]:
X, y_labels = make_blobs(
    n_samples=300,
    centers=4,
    cluster_std=0.75,
    random_state=42
)

plt.scatter(X[:, 0], X[:, 1], s=30)
plt.title("Dados Originais")
plt.show()



Funções para gerar uma visualização gráfica dos dados. Cada ponto representa uma amostra, e as cores indicam os grupos ou classes atribuídos.

In [ ]:
def euclidean_distance(a, b):
    return np.sqrt(((a - b) ** 2).sum(axis=1))

def assign_clusters(X, centroids):
    distances = np.column_stack([
        euclidean_distance(X, c) for c in centroids
    ])
    return np.argmin(distances, axis=1)

def update_centroids(X, labels, k):
    return np.array([
        X[labels == i].mean(axis=0)
        for i in range(k)
    ])

def plot_step(X, labels, centroids, title):
    plt.figure(figsize=(6, 5))
    plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=35)
    plt.scatter(
        centroids[:, 0], centroids[:, 1],
        c="black", marker="X", s=250, label="Centróides"
    )
    plt.title(title)
    plt.legend()
    plt.show()

### K-Means Passo a Passo


Infere os centróides de maneira randomica e itera para demonstrar a evolução da clusterização.

In [ ]:

k = 4
max_iter = 10

# Inicialização aleatória
indices = np.random.choice(len(X), k, replace=False)
centroids = X[indices]

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], s=35)
plt.scatter(
    centroids[:, 0], centroids[:, 1],
    c="red", marker="X", s=250, label="Centróides iniciais"
)
plt.title("Inicialização dos centróides")
plt.legend()
plt.show()

for iteration in range(1, max_iter + 1):
    labels = assign_clusters(X, centroids)
    new_centroids = update_centroids(X, labels, k)

    plot_step(X, labels, new_centroids, f"Iteração {iteration}")

    if np.allclose(centroids, new_centroids):
        print(f"Convergência atingida na iteração {iteration}.")
        break

    centroids = new_centroids

clustering_metrics(X, labels, y_labels)

### K-means com Sklearn

Instancia e executa o algoritmo K-Means. O parâmetro `n_clusters` define quantos grupos serão formados. `n_init=10` indica que o algoritmo será executado dez vezes com inicializações diferentes, mantendo a melhor solução em termos de inércia.

In [ ]:
model = KMeans(n_clusters=k, random_state=42, n_init=10)
labels_final = model.fit_predict(X)

plot_step(X, labels_final, model.cluster_centers_, "Resultado Final")

print("Inércia:", model.inertia_)
print("Silhouette Score:", silhouette_score(X, labels_final))

### Clusterização/Segmentação de imagens

Carregar imagem de exemplo do scikit-learn

Opções disponíveis:

 - "china.jpg"
 - "flower.jpg"

In [ ]:
image = load_sample_image("china.jpg")
image = img_as_float(image)   # converte para float no intervalo [0, 1]

print("Shape da imagem:", image.shape)

plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.title("Imagem Original")
plt.axis("off")
plt.show()


Instancia e executa o algoritmo K-Means. O parâmetro `n_clusters` define quantos grupos serão formados. `n_init=10` indica que o algoritmo será executado dez vezes com inicializações diferentes, mantendo a melhor solução em termos de inércia.

In [ ]:
def kmeans_image_segmentation(image, k=8, random_state=42):
    h, w, c = image.shape

    # Cada pixel é um vetor RGB
    pixels = image.reshape(-1, 3)

    # Ajuste do K-Means
    kmeans = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=10
    )

    labels = kmeans.fit_predict(pixels)
    centers = kmeans.cluster_centers_

    # Reconstrução da imagem
    segmented = centers[labels].reshape(h, w, 3)

    # Garante valores válidos
    segmented = np.clip(segmented, 0, 1)

    return segmented, labels, centers


In [ ]:
for k in [2, 4, 8, 16, 32]:
    segmented, labels, centers = kmeans_image_segmentation(image, k=k)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("Imagem Original")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(segmented)
    plt.title(f"K-Means Segmentado (K = {k})")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    print(f"K = {k}")
    print(f"Número de cores finais: {len(centers)}")

#### Interpretação

##### Em dados sintéticos

O algoritmo:

1. Escolhe centróides iniciais;
2. Atribui cada ponto ao centróide mais próximo;
3. Recalcula os centróides;
4. Repete até convergir.

##### Em imagens
Cada pixel é tratado como um vetor RGB:

\[
[R, G, B]
\]

O K-Means agrupa pixels com cores semelhantes e substitui cada pixel pela cor do centróide do seu cluster.

#### Aplicações
- Segmentação de imagens
- Compressão de cores
- Quantização
- Pré-processamento para visão computacional


## Clustering Hierárquico

É mostrado:

1. Dados sintéticos antes do agrupamento.
2. Construção do dendrograma.
3. Formação progressiva dos clusters.
4. Resultado final.
5. Aplicação em um problema real: segmentação de clientes.


### Introdução

O Clustering Hierárquico constrói uma árvore de agrupamentos, chamada dendrograma. Em vez de definir previamente o número de clusters, o algoritmo vai unindo os grupos mais semelhantes até restar apenas um grande grupo.

A principal vantagem é permitir visualizar os dados em diferentes níveis de granularidade.

Neste exemplo, você verá:
- como interpretar um dendrograma;
- como escolher o número de clusters com base no corte do dendrograma;
- uma aplicação no dataset Wine.

**Aplicações práticas:** bioinformática, taxonomia, segmentação de clientes e organização de documentos.

#### Funções auxiliares

Somente para plotar a inferência.

In [ ]:
def plot_clusters(X, labels, title):
    plt.figure(figsize=(6, 5))
    plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=40)
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()


def plot_original_data(X, title):
    plt.figure(figsize=(6, 5))
    plt.scatter(X[:, 0], X[:, 1], color="gray", s=40)
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()

#### Exemplo Didático com Dados Sintéticos

Gera um conjunto de dados sintético utilizando `make_blobs()`. Cada blob representa um grupo de pontos distribuídos ao redor de um centróide. Esse tipo de dataset é ideal para demonstrar algoritmos de clustering, pois a estrutura dos grupos é conhecida previamente.

In [ ]:
print("Exemplo simples com dados sintéticos")

# Geração de dados
X, y_true = make_blobs(
    n_samples=300,
    centers=4,
    cluster_std=0.70,
    random_state=42
)

# Dados antes do agrupamento
plot_original_data(X, "Dados Originais (Sem Rótulos)")

#### Dendrograma

In [ ]:
print("Calculando a matriz de linkage...")

Z = linkage(X, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z)
plt.title("Dendrograma Completo")
plt.xlabel("Amostras")
plt.ylabel("Distância")
plt.show()

In [ ]:
# Dendrograma resumido
plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="lastp", p=20)
plt.title("Dendrograma Resumido")
plt.xlabel("Clusters")
plt.ylabel("Distância")
plt.show()

#### Formação progressiva dos clusters

É aplicado o Clustering Hierárquico Aglomerativo. Cada amostra começa como um cluster individual e, a cada etapa, os clusters mais semelhantes são unidos até restar o número desejado.

In [ ]:
print("Visualizando a formação dos clusters para diferentes valores de K")

for k in [2, 3, 4, 5]:
    model = AgglomerativeClustering(
        n_clusters=k,
        linkage="ward"
    )
    labels = model.fit_predict(X)

    plot_clusters(X, labels, f"Agglomerative Clustering (K={k})")

In [ ]:
# Resultado final

k_final = 4
model = AgglomerativeClustering(
    n_clusters=k_final,
    linkage="ward"
)

labels_final = model.fit_predict(X)

plot_clusters(X, labels_final, f"Resultado Final (K={k_final})")

### Aplicação Real: Segmentação de Vinhos

Nesta etapa os dados são padronizados com `StandardScaler()`. Cada variável passa a ter média 0 e desvio padrão 1. Isso é fundamental porque muitos algoritmos utilizam distâncias e podem ser influenciados por variáveis em escalas diferentes.

In [ ]:
print("Aplicação real: Dataset Wine")

# Dataset real
wine = load_wine()
X_real = wine.data
y_real = wine.target

print("Shape do dataset:", X_real.shape)
print("Número de classes reais:", len(np.unique(y_real)))

# Padronização
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_real)

# Redução para visualização
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Dados antes do agrupamento
plot_original_data(
    X_pca,
    "Wine Dataset em 2D (PCA) - Antes do Clustering"
)

# Dendrograma
Z_real = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z_real, truncate_mode="lastp", p=30)
plt.title("Dendrograma - Wine Dataset")
plt.xlabel("Clusters")
plt.ylabel("Distância")
plt.show()

# Clustering
model_real = AgglomerativeClustering(
    n_clusters=3,
    linkage="ward"
)

labels_real = model_real.fit_predict(X_scaled)

# Resultado em PCA
plot_clusters(
    X_pca,
    labels_real,
    "Wine Dataset - Clustering Hierárquico"
)

# Comparação com classes reais
plot_clusters(
    X_pca,
    y_real,
    "Wine Dataset - Classes Reais"
)

### Métricas

Aqui são calculadas as métricas de avaliação. Essas métricas permitem comparar a qualidade dos agrupamentos.

In [ ]:
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    davies_bouldin_score,
)

print("Métricas de Avaliação")

print("Silhouette Score:",
      silhouette_score(X_scaled, labels_real))

print("Davies-Bouldin:",
      davies_bouldin_score(X_scaled, labels_real))

print("Adjusted Rand Index:",
      adjusted_rand_score(y_real, labels_real))

## DBSCAN

É mostrado:

1. Dados antes da clusterização.
2. Construção progressiva dos clusters.
3. Identificação de pontos centrais, de borda e ruído.
4. Resultado final.
5. Aplicação em um problema real: detecção de anomalias em transações.


### Introdução 

O DBSCAN (*Density-Based Spatial Clustering of Applications with Noise*) agrupa pontos com base em regiões densas. Ele é especialmente útil quando:
- os clusters têm formatos irregulares;
- há ruído ou outliers;
- o número de clusters é desconhecido.

Diferente do K-Means, o DBSCAN pode marcar pontos isolados como ruído (`-1`).

Neste exemplo, você verá:
- o papel dos parâmetros `eps` e `min_samples`;
- a identificação de pontos centrais, de borda e ruído;
- uma aplicação em um conjunto de dados real.

**Aplicações práticas:** GPS, detecção de fraudes, análise espacial e dados com outliers.

#### Funções auxiliares

In [ ]:
def plot_points(X, labels=None, title="Dados"):
    plt.figure(figsize=(6, 5))

    if labels is None:
        plt.scatter(X[:, 0], X[:, 1], color="gray", s=40)
    else:
        plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=40)

    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()


def plot_dbscan_step(X, labels, core_mask, title):
    plt.figure(figsize=(7, 6))

    # Ruído
    noise = labels == -1
    plt.scatter(
        X[noise, 0], X[noise, 1],
        c="black", marker="x", s=80, label="Ruído"
    )

    # Pontos centrais
    core_labels = labels.copy()
    core_labels[~core_mask] = -1
    valid = core_labels != -1
    plt.scatter(
        X[valid, 0], X[valid, 1],
        c=core_labels[valid], cmap="tab10",
        s=120, marker="o", edgecolors="k",
        label="Core Points"
    )

    # Pontos de borda
    border_mask = (labels != -1) & (~core_mask)
    plt.scatter(
        X[border_mask, 0], X[border_mask, 1],
        c=labels[border_mask], cmap="tab10",
        s=60, marker="s",
        label="Border Points"
    )

    plt.title(title)
    plt.legend()
    plt.show()

### Exemplo Didático com make_moons

Aqui é aplicado o DBSCAN. O parâmetro `eps` define o raio de vizinhança e `min_samples` indica o número mínimo de pontos necessários para formar uma região densa. Pontos classificados como `-1` são considerados ruído.

Para iniciar, um conjunto de dados no formato de duas luas entrelaçadas. Esse dataset é importante porque apresenta uma estrutura não linear, na qual algoritmos como K-Means têm dificuldade, enquanto DBSCAN e Spectral Clustering costumam obter bons resultados.

In [ ]:
print("Exemplo simples com make_moons")

X, y_true = make_moons(
    n_samples=400,
    noise=0.05,
    random_state=42
)

X = StandardScaler().fit_transform(X)

# Dados originais
plot_points(X, title="Dados Originais (Sem Rótulos)")


# Teste com diferentes valores de eps
for eps in [0.15, 0.20, 0.25, 0.30]:
    model = DBSCAN(eps=eps, min_samples=5)
    labels = model.fit_predict(X)

    core_mask = np.zeros_like(labels, dtype=bool)
    core_mask[model.core_sample_indices_] = True

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = np.sum(labels == -1)

    print(f"eps = {eps}")
    print(f"Número de clusters: {n_clusters}")
    print(f"Pontos de ruído: {n_noise}")

    plot_dbscan_step(
        X,
        labels,
        core_mask,
        title=f"DBSCAN (eps={eps}, min_samples=5)"
    )


# Resultado final
eps_final = 0.25
model = DBSCAN(eps=eps_final, min_samples=5)
labels_final = model.fit_predict(X)

core_mask = np.zeros_like(labels_final, dtype=bool)
core_mask[model.core_sample_indices_] = True

plot_dbscan_step(
    X,
    labels_final,
    core_mask,
    title="Resultado Final do DBSCAN"
)

### Aplicação Real: Detecção de Anomalias em Wine Dataset

Aqui os dados são padronizados com `StandardScaler()`. Cada variável passa a ter média 0 e desvio padrão 1. Isso é fundamental porque muitos algoritmos utilizam distâncias e podem ser influenciados por variáveis em escalas diferentes.

In [ ]:
print("Aplicação real: Detecção de anomalias")

wine = load_wine()
X_real = wine.data
y_real = wine.target

# Padronização
X_scaled = StandardScaler().fit_transform(X_real)

# Redução para visualização
X_pca = PCA(n_components=2).fit_transform(X_scaled)

# Dados antes
plot_points(
    X_pca,
    title="Wine Dataset em PCA (Antes do DBSCAN)"
)

# Aplicação do DBSCAN
model_real = DBSCAN(eps=1.2, min_samples=5)
labels_real = model_real.fit_predict(X_scaled)

core_mask_real = np.zeros_like(labels_real, dtype=bool)
core_mask_real[model_real.core_sample_indices_] = True

plot_dbscan_step(
    X_pca,
    labels_real,
    core_mask_real,
    title="DBSCAN no Wine Dataset"
)

# Classes reais
plot_points(
    X_pca,
    labels=y_real,
    title="Classes Reais do Wine Dataset"
)

### Métricas

In [ ]:
valid = labels_real != -1

if len(np.unique(labels_real[valid])) > 1:
    print("Silhouette Score:",
          silhouette_score(X_scaled[valid], labels_real[valid]))

    print("Davies-Bouldin:",
          davies_bouldin_score(X_scaled[valid], labels_real[valid]))

print("Adjusted Rand Index:",
      adjusted_rand_score(y_real, labels_real))

print("Número de anomalias detectadas:",
      np.sum(labels_real == -1))

## Gaussian Mixture Models

É mostrado:
1. Dados antes da clusterização.
2. Inicialização dos componentes Gaussianos.
3. Iterações do algoritmo EM (Expectation-Maximization).
4. Evolução das probabilidades de pertencimento.
5. Resultado final.
6. Aplicação em dados reais.


### Introdução

O Gaussian Mixture Model (GMM) assume que os dados foram gerados por uma combinação de várias distribuições Gaussianas.

Ao contrário do K-Means, que atribui cada ponto a um único cluster, o GMM fornece **probabilidades de pertencimento**. Um mesmo ponto pode pertencer parcialmente a vários clusters.

O ajuste do modelo é realizado pelo algoritmo EM (*Expectation-Maximization*).

Neste exemplo, você verá:
- a evolução do algoritmo EM;
- visualização das elipses das Gaussianas;
- probabilidades de pertencimento.

**Aplicações práticas:** modelagem de densidade, segmentação de clientes, reconhecimento de padrões e detecção de anomalias.

### Funções auxiliares

In [ ]:
def plot_original(X, title):
    plt.figure(figsize=(6, 5))
    plt.scatter(X[:, 0], X[:, 1], color="gray", s=35)
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()


def draw_ellipse(position, covariance, ax=None, **kwargs):
    """
    Desenha elipses representando as Gaussianas.
    """
    ax = ax or plt.gca()

    if covariance.shape == (2, 2):
        U, s, Vt = np.linalg.svd(covariance)
        angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
        width, height = 2 * np.sqrt(s)
    else:
        angle = 0
        width, height = 2 * np.sqrt(covariance)

    for nsig in [1, 2, 3]:
        ax.add_patch(
            Ellipse(
                position,
                nsig * width,
                nsig * height,
                angle=angle,
                fill=False,
                lw=2,
                **kwargs
            )
        )


def plot_gmm(X, labels, means, covariances, title):
    fig, ax = plt.subplots(figsize=(7, 6))

    ax.scatter(
        X[:, 0],
        X[:, 1],
        c=labels,
        cmap="tab10",
        s=35,
        alpha=0.7
    )

    for mean, cov in zip(means, covariances):
        draw_ellipse(mean, cov, ax=ax, edgecolor="black")

    ax.scatter(
        means[:, 0],
        means[:, 1],
        c="red",
        marker="X",
        s=200,
        label="Médias"
    )

    ax.set_title(title)
    ax.legend()
    plt.show()

### Exemplo Didático com Dados Sintéticos

Aqui é ajustado um Gaussian Mixture Model (GMM). Cada cluster é modelado como uma distribuição Gaussiana multivariada. O método `fit_predict()` retorna o cluster mais provável para cada amostra. Gera-se um conjunto de dados sintético utilizando `make_blobs()`. Cada blob representa um grupo de pontos distribuídos ao redor de um centróide. Esse tipo de dataset é ideal para demonstrar algoritmos de clustering, pois a estrutura dos grupos é conhecida previamente.

In [ ]:
print("Exemplo simples com dados sintéticos")

X, y_true = make_blobs(
    n_samples=500,
    centers=4,
    cluster_std=[0.5, 1.2, 0.8, 1.0],
    random_state=42
)

# Transformação linear para criar clusters elípticos
transform = np.array([[0.6, -0.6],
                      [-0.4, 0.8]])
X = X @ transform

plot_original(X, "Dados Originais")


# Evolução do EM
print("Visualizando a evolução do algoritmo EM")

for n_iter in [1, 2, 5, 10, 20]:
    gmm = GaussianMixture(
        n_components=4,
        covariance_type="full",
        max_iter=n_iter,
        random_state=42,
        init_params="kmeans"
    )

    gmm.fit(X)
    labels = gmm.predict(X)

    print(f"Iterações EM = {n_iter}")
    print("Log-likelihood:", gmm.lower_bound_)

    plot_gmm(
        X,
        labels,
        gmm.means_,
        gmm.covariances_,
        f"GMM após {n_iter} iterações EM"
    )


# Resultado final
gmm_final = GaussianMixture(
    n_components=4,
    covariance_type="full",
    random_state=42
)

gmm_final.fit(X)
labels_final = gmm_final.predict(X)

plot_gmm(
    X,
    labels_final,
    gmm_final.means_,
    gmm_final.covariances_,
    "Resultado Final do GMM"
)

# Probabilidades soft
probs = gmm_final.predict_proba(X)
print("Probabilidades de pertencimento das 5 primeiras amostras:")
print(np.round(probs[:5], 3))

### Aplicação Real: Segmentação de Vinhos

Os dados são padronizados com `StandardScaler()`. Cada variável passa a ter média 0 e desvio padrão 1. Isso é fundamental porque muitos algoritmos utilizam distâncias e podem ser influenciados por variáveis em escalas diferentes.

In [ ]:
print("Aplicação real: DATASET WINE")

wine = load_wine()
X_real = wine.data
y_real = wine.target

# Padronização
X_scaled = StandardScaler().fit_transform(X_real)

# Redução para visualização
X_pca = PCA(n_components=2).fit_transform(X_scaled)

# Dados antes
plot_original(X_pca, "Wine Dataset em PCA (Antes do GMM)")

# Aplicação do GMM
gmm_real = GaussianMixture(
    n_components=3,
    covariance_type="full",
    random_state=42
)

gmm_real.fit(X_scaled)
labels_real = gmm_real.predict(X_scaled)

plot_gmm(
    X_pca,
    labels_real,
    gmm_real.means_[:, :2],     # projeção aproximada das médias
    [np.eye(2)] * 3,            # simplificação visual
    "Wine Dataset - Gaussian Mixture Model"
)

# Classes reais
plot_original(X_pca, "Visualização PCA")
plt.figure(figsize=(6, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_real, cmap="tab10", s=35)
plt.title("Classes Reais do Wine Dataset")
plt.show()

### Métricas

In [ ]:
print("Métricas")

print("Silhouette Score:",
      silhouette_score(X_scaled, labels_real))

print("Davies-Bouldin:",
      davies_bouldin_score(X_scaled, labels_real))

print("Adjusted Rand Index:",
      adjusted_rand_score(y_real, labels_real))

print("Pesos das Gaussianas:")
print(np.round(gmm_real.weights_, 3))

## Spectral Clustering

É mostrado:
1. Dados antes da clusterização.
2. Construção do grafo de similaridade.
3. Matriz de afinidade.
4. Laplaciana do grafo.
5. Embedding espectral.
6. Clusterização final no espaço espectral.
7. Aplicação em dados reais.


### Introdução

O Spectral Clustering é uma técnica poderosa para dados com estruturas complexas e não lineares. Ele transforma os dados em um grafo de similaridade, calcula um embedding espectral e, em seguida, aplica K-Means nesse novo espaço.

Esse método costuma funcionar muito bem quando algoritmos tradicionais falham em separar clusters com formatos não convexos.

Neste exemplo, você verá:
- a construção da matriz de similaridade;
- o embedding espectral;
- a clusterização final.

**Aplicações práticas:** segmentação de imagens, análise de redes e agrupamento de padrões complexos.

### Funções auxiliares

In [ ]:
def plot_points(X, labels=None, title="Dados"):
    plt.figure(figsize=(6, 5))

    if labels is None:
        plt.scatter(X[:, 0], X[:, 1], color="gray", s=35)
    else:
        plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=35)

    plt.title(title)
    plt.xlabel("Dimensão 1")
    plt.ylabel("Dimensão 2")
    plt.show()

### Exemplo Didático com make_moons

Aqui aplicamos Spectral Clustering. Primeiro, é construído um grafo de similaridade entre as amostras. Em seguida, os dados são projetados em um espaço espectral e o K-Means é aplicado nesse novo espaço. É gerado um conjunto de dados no formato de duas luas entrelaçadas. Esse dataset é importante porque apresenta uma estrutura não linear, na qual algoritmos como K-Means têm dificuldade, enquanto DBSCAN e Spectral Clustering costumam obter bons resultados.

In [ ]:
print("Exemplo simples com dados sintéticos make_moons")

X, y_true = make_moons(
    n_samples=400,
    noise=0.05,
    random_state=42
)

X = StandardScaler().fit_transform(X)

# Dados originais
plot_points(X, title="Dados Originais")

# Matriz de Afinidade (RBF)
gamma = 1.0
A = rbf_kernel(X, gamma=gamma)

plt.figure(figsize=(6, 5))
sns.heatmap(A, cmap="viridis")
plt.title("Matriz de Afinidade (RBF Kernel)")
plt.show()

# Grafo k-NN
W = kneighbors_graph(
    X,
    n_neighbors=10,
    include_self=True
).toarray()

plt.figure(figsize=(6, 5))
sns.heatmap(W, cmap="viridis")
plt.title("Grafo k-NN")
plt.show()

# Embedding Espectral
embedding = SpectralEmbedding(
    n_components=2,
    affinity="nearest_neighbors",
    n_neighbors=10
)

X_spec = embedding.fit_transform(X)

plot_points(
    X_spec,
    title="Embedding Espectral"
)

# K-Means no espaço espectral
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_emb = kmeans.fit_predict(X_spec)

plot_points(
    X_spec,
    labels_emb,
    title="K-Means no Espaço Espectral"
)

# Resultado Final
model = SpectralClustering(
    n_clusters=2,
    affinity="nearest_neighbors",
    n_neighbors=10,
    random_state=42
)

labels_final = model.fit_predict(X)

plot_points(
    X,
    labels_final,
    title="Resultado Final do Spectral Clustering"
)

### Aplicação Real: Segmentação de Dígitos

Aqui os dados são padronizados com `StandardScaler()`. Cada variável passa a ter média 0 e desvio padrão 1. Isso é fundamental porque muitos algoritmos utilizam distâncias e podem ser influenciados por variáveis em escalas diferentes.

In [ ]:
print("Aplicação real: DATASET DIGITS")

digits = load_digits()
X_real = digits.data
y_real = digits.target

# Padronização
X_scaled = StandardScaler().fit_transform(X_real)

# Visualização PCA
X_pca = PCA(n_components=2).fit_transform(X_scaled)

plot_points(
    X_pca,
    title="Digits Dataset em PCA (Antes do Clustering)"
)

# Usaremos apenas 10 classes (0–9)
model_real = SpectralClustering(
    n_clusters=10,
    affinity="nearest_neighbors",
    n_neighbors=15,
    random_state=42
)

labels_real = model_real.fit_predict(X_scaled)

plot_points(
    X_pca,
    labels_real,
    title="Digits Dataset - Spectral Clustering"
)

plot_points(
    X_pca,
    y_real,
    title="Digits Dataset - Classes Reais"
)

### Métricas

In [ ]:
print("Métricas")

print("Silhouette Score:",
      silhouette_score(X_scaled, labels_real))

print("Davies-Bouldin:",
      davies_bouldin_score(X_scaled, labels_real))

print("Adjusted Rand Index:",
      adjusted_rand_score(y_real, labels_real))

## PCA

É mostrado:

1. Mostrar os dados originais.
2. Padronizar os atributos.
3. Calcular os componentes principais.
4. Visualizar os autovetores.
5. Mostrar a variância explicada.
6. Projetar os dados em menor dimensão.
7. Aplicar PCA em um dataset real.


### Introdução 

O PCA (*Principal Component Analysis*) é uma técnica de redução de dimensionalidade. Ele transforma as variáveis originais em novos eixos chamados componentes principais, que preservam a maior parte da variância dos dados.

O PCA é muito utilizado como etapa de pré-processamento e visualização.

Neste exemplo, você verá:
- os autovetores e autovalores;
- a variância explicada por cada componente;
- a projeção dos dados em 2 dimensões.

**Aplicações práticas:** visualização, compressão de dados, remoção de ruído e preparação para clustering.

### Funções auxiliares

In [ ]:
def plot_points(X, labels=None, title="Dados"):
    plt.figure(figsize=(6, 5))

    if labels is None:
        plt.scatter(X[:, 0], X[:, 1], color="gray", s=35)
    else:
        plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=35)

    plt.title(title)
    plt.xlabel("Dimensão 1")
    plt.ylabel("Dimensão 2")
    plt.show()

### Exemplo Didático em 2D

In [ ]:
print("Exemplo simples com dados sintéticos")

# Geração de dados correlacionados
rng = np.random.RandomState(42)
X = rng.randn(500, 2)
X[:, 1] = 0.8 * X[:, 0] + 0.2 * rng.randn(500)

# Dados originais
plot_points(X, title="Dados Originais")

# Padronização
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# Ajuste do PCA
pca = PCA(n_components=2)
pca.fit(X_scaled)

print("Componentes Principais (autovetores):")
print(pca.components_)

print("\nAutovalores (variância):")
print(pca.explained_variance_)

print("\nRazão de variância explicada:")
print(pca.explained_variance_ratio_)

# Visualização dos componentes principais
plt.figure(figsize=(6, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], alpha=0.3)

origin = np.zeros(2)

for i, vector in enumerate(pca.components_):
    plt.arrow(
        origin[0],
        origin[1],
        vector[0] * 3,
        vector[1] * 3,
        width=0.03,
        label=f"PC{i+1}"
    )

plt.legend()
plt.title("Componentes Principais")
plt.xlabel("X1")
plt.ylabel("X2")
plt.axis("equal")
plt.show()

In [ ]:
# Projeção para 1 componente
pca_1d = PCA(n_components=1)
X_1d = pca_1d.fit_transform(X_scaled)
X_reconstructed = pca_1d.inverse_transform(X_1d)

plt.figure(figsize=(6, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1],
            alpha=0.3, label="Original")
plt.scatter(X_reconstructed[:, 0],
            X_reconstructed[:, 1],
            alpha=0.6, label="Reconstruído (1 PC)")
plt.legend()
plt.title("Redução para 1 Componente Principal")
plt.show()

In [ ]:
# Variância explicada acumulada
plt.figure(figsize=(6, 4))
plt.plot(
    np.cumsum(pca.explained_variance_ratio_),
    marker="o"
)
plt.ylim(0, 1.05)
plt.xlabel("Número de Componentes")
plt.ylabel("Variância Explicada Acumulada")
plt.title("Variância Explicada Acumulada")
plt.show()

### Aplicação Real: Wine Dataset

In [ ]:
print("Aplicação real:  WINE DATASET")

wine = load_wine()
X_real = wine.data
y_real = wine.target
feature_names = wine.feature_names

print("Dimensão original:", X_real.shape)

# Padronização
scaler = StandardScaler()
X_scaled_real = scaler.fit_transform(X_real)

# PCA para 2 componentes
pca_real = PCA(n_components=2)
X_pca = pca_real.fit_transform(X_scaled_real)

print("Dimensão após PCA:", X_pca.shape)
print("Variância explicada:",
      pca_real.explained_variance_ratio_)

# Dados antes (usando duas primeiras features)
plot_points(
    X_scaled_real[:, :2],
    title="Wine Dataset (2 primeiras variáveis)"
)

# Dados após PCA
plot_points(
    X_pca,
    labels=y_real,
    title="Wine Dataset após PCA"
)

In [ ]:
# Importância das variáveis (loadings)
loadings = pd.DataFrame(
    pca_real.components_.T,
    columns=["PC1", "PC2"],
    index=feature_names
)

print("\nLoadings:")
print(loadings)

# Heatmap dos loadings
plt.figure(figsize=(8, 6))
sns.heatmap(loadings, annot=True, cmap="coolwarm", center=0)
plt.title("Loadings das Variáveis")
plt.show()

## Detecção de Anomalias - Isolation Forest

É mostrado:

1. Mostrar os dados antes da detecção.
2. Explicar a ideia de isolamento por árvores aleatórias.
3. Ajustar o modelo.
4. Visualizar scores de anomalia.
5. Mostrar os outliers detectados.
6. Aplicar em um dataset real.


### Introdução 

O Isolation Forest é um algoritmo de detecção de anomalias que funciona isolando observações por meio de divisões aleatórias em árvores.

A intuição é simples:
- pontos normais precisam de muitas divisões para serem isolados;
- anomalias são isoladas rapidamente.

Neste exemplo, você verá:
- dados sintéticos com outliers;
- scores de anomalia;
- visualização das observações consideradas anômalas.

**Aplicações práticas:** fraudes financeiras, falhas industriais, intrusão em redes e monitoramento de sensores.

### Funções auxiliares

In [ ]:
def plot_points(X, labels=None, title="Dados"):
    plt.figure(figsize=(6, 5))

    if labels is None:
        plt.scatter(X[:, 0], X[:, 1], color="gray", s=35)
    else:
        # 1 = normal, -1 = anomalia
        colors = np.where(labels == -1, "red", "steelblue")
        plt.scatter(X[:, 0], X[:, 1], c=colors, s=40)

    plt.title(title)
    plt.xlabel("Dimensão 1")
    plt.ylabel("Dimensão 2")
    plt.show()

### Exemplo Didático com Dados Sintéticos

Gera-se um conjunto de dados sintético utilizando `make_blobs()`. Cada blob representa um grupo de pontos distribuídos ao redor de um centróide. Esse tipo de dataset é ideal para demonstrar algoritmos de clustering, pois a estrutura dos grupos é conhecida previamente.

In [ ]:
print("Exemplo simples com dados sintéticos")

# Dados normais
X_normal, _ = make_blobs(
    n_samples=300,
    centers=2,
    cluster_std=0.8,
    random_state=42
)

# Anomalias artificiais
rng = np.random.RandomState(42)
X_outliers = rng.uniform(low=-8, high=8, size=(20, 2))

# Dataset completo
X = np.vstack([X_normal, X_outliers])

# Ground truth (apenas para visualização)
y_true = np.hstack([
    np.ones(len(X_normal)),
    -np.ones(len(X_outliers))
])

# Dados originais
plot_points(X, title="Dados Originais")



Aqui é ajustado o Isolation Forest. O algoritmo constrói várias árvores aleatórias e mede quantas divisões são necessárias para isolar cada ponto.

In [ ]:
# Ajuste do Isolation Forest
model = IsolationForest(
    n_estimators=100,
    contamination=0.06,
    random_state=42
)

model.fit(X)

# Predições
labels = model.predict(X)  # 1 = normal, -1 = anomalia
scores = model.decision_function(X)

# Visualização final
plot_points(X, labels, "Isolation Forest - Anomalias Detectadas")


In [ ]:
# Histograma dos scores
plt.figure(figsize=(6, 4))
plt.hist(scores, bins=30)
plt.title("Distribuição dos Scores de Anomalia")
plt.xlabel("Decision Function")
plt.ylabel("Frequência")
plt.show()



In [ ]:
# Curvas de nível
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 200),
    np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 200)
)

grid = np.c_[xx.ravel(), yy.ravel()]
Z = model.decision_function(grid).reshape(xx.shape)

plt.figure(figsize=(7, 6))
plt.contourf(xx, yy, Z, levels=50, cmap="RdYlBu")
plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors="black")
plot_colors = np.where(labels == -1, "red", "white")
plt.scatter(X[:, 0], X[:, 1], c=plot_colors, edgecolors="k")
plt.title("Fronteira de Decisão do Isolation Forest")
plt.show()

print("Número de anomalias detectadas:", np.sum(labels == -1))

### Aplicação Real: Wine Dataset

In [ ]:
print("Aplicação real: WINE DATASET")

wine = load_wine()
X_real = wine.data
feature_names = wine.feature_names

# Padronização
X_scaled = StandardScaler().fit_transform(X_real)

# PCA para visualização
X_pca = PCA(n_components=2).fit_transform(X_scaled)

# Dados antes
plot_points(
    X_pca,
    title="Wine Dataset em PCA (Antes da Detecção)"
)

# Aplicação do Isolation Forest
iso_real = IsolationForest(
    contamination=0.05,
    random_state=42
)

labels_real = iso_real.fit_predict(X_scaled)
scores_real = iso_real.decision_function(X_scaled)

# Resultado
plot_points(
    X_pca,
    labels_real,
    title="Wine Dataset - Anomalias Detectadas"
)

# Amostras mais anômalas
df_scores = pd.DataFrame({
    "score": scores_real,
    "label": labels_real
})

print("Amostras mais anômalas:")
print(df_scores.sort_values("score").head(10))